In [118]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

# =========================
# INIT
# =========================
fake = Faker("en_IN")
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# =========================
# CONFIG
# =========================
NUM_CUSTOMERS = 8000
TARGET_ACCOUNTS = 10000
TARGET_TRANSACTIONS = 1_000_000

TX_START = datetime(2025, 7, 1)
TX_END   = datetime(2026, 6, 30)

# =========================
# CUSTOMER INCOME MAP
# =========================
INCOME_MAP = {
    "<5L": 30000,
    "5-10L": 70000,
    "10-25L": 150000,
    ">25L": 400000
}

# =========================
# 1. CUSTOMERS
# =========================
customer_ids = [f"CUST_{i:05d}" for i in range(1, NUM_CUSTOMERS + 1)]

df_customers = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_type": np.random.choice(
        ["INDIVIDUAL", "CORPORATE", "NPO", "NRI"],
        NUM_CUSTOMERS,
        p=[0.85, 0.10, 0.03, 0.02]
    ),
    "income_bracket": np.random.choice(
        ["<5L", "5-10L", "10-25L", ">25L"],
        NUM_CUSTOMERS,
        p=[0.60, 0.25, 0.10, 0.05]
    ),
    "customer_risk_rating": np.random.choice(
        ["LOW", "MEDIUM", "HIGH"],
        NUM_CUSTOMERS,
        p=[0.72, 0.23, 0.05]
    ),
})

df_customers["monthly_income"] = df_customers["income_bracket"].map(INCOME_MAP)
df_customers["customer_segment"] = np.where(
    df_customers["customer_type"] == "CORPORATE", "CORPORATE", "RETAIL"
)

# =========================
# 2. ACCOUNTS
# =========================
accounts = []
acc_id = 1

for _, c in df_customers.iterrows():
    for _ in range(np.random.choice([1,2,3], p=[0.8,0.15,0.05])):
        if acc_id > TARGET_ACCOUNTS:
            break

        open_date = fake.date_between(datetime(2000,1,1), datetime(2024,12,31))
        status = np.random.choice(["ACTIVE","DORMANT","INACTIVE"], p=[0.85,0.10,0.05])

        last_dormant = None
        close_date = None

        if status == "DORMANT":
            last_dormant = fake.date_between(open_date, TX_END)
        if status == "INACTIVE":
            close_date = fake.date_between(open_date, TX_END)

        accounts.append({
            "account_id": f"ACC_{acc_id:06d}",
            "customer_id": c.customer_id,
            "account_type": "CURRENT" if c.customer_segment=="CORPORATE" else "SAVINGS",
            "account_open_date": open_date,
            "account_status": status,
            "last_dormant_date": last_dormant,
            "account_close_date": close_date
        })
        acc_id += 1

df_accounts = pd.DataFrame(accounts)

# =========================
# 3. TRANSACTIONS
# =========================
tx_rows = []
tx_id = 1

accounts_joined = df_accounts.merge(
    df_customers[["customer_id","monthly_income","customer_segment"]],
    on="customer_id"
)

while len(tx_rows) < TARGET_TRANSACTIONS:
    acc = accounts_joined.sample(1).iloc[0]

    start = max(TX_START.date(), acc.account_open_date)
    end = TX_END.date()

    if pd.notna(acc.account_close_date):
        end = min(end, acc.account_close_date)
    if pd.notna(acc.last_dormant_date):
        end = min(end, acc.last_dormant_date - timedelta(days=180))

    if start >= end:
        continue

    tx_date = start + timedelta(days=random.randint(0, (end-start).days))

    is_cash = random.random() < 0.05

    if is_cash:
        amount = np.random.lognormal(mean=9, sigma=0.5)   # mostly <50k
        amount = min(amount, 50000)
        category = "CASH"
    else:
        base = acc.monthly_income * np.random.uniform(0.3, 3)
        amount = np.random.lognormal(mean=np.log(base), sigma=1.0)
        category = random.choice(
            ["NEFT","RTGS","IMPS","UPI","CHEQUE","INTERNATIONAL_WIRE","FX"]
        )

    tx_rows.append({
        "transaction_id": f"TXN_{tx_id:09d}",
        "account_id": acc.account_id,
        "customer_id": acc.customer_id,
        "transaction_datetime": tx_date,
        "transaction_type": random.choice(["DEBIT","CREDIT"]),
        "transaction_category": category,
        "transaction_amount": round(amount,2)
    })

    tx_id += 1

df_transactions = pd.DataFrame(tx_rows)
df_customers.to_csv(r"E:\VS code stuff\Banks Data\very_large_bank\customers.csv", index=False)
df_accounts.to_csv(r"E:\VS code stuff\Banks Data\very_large_bank\accounts.csv", index=False)
df_transactions.to_csv(r"E:\VS code stuff\Banks Data\very_large_bank\transactions.csv", index=False)
print("Customers:", len(df_customers))
print("Accounts:", len(df_accounts))
print("Transactions:", len(df_transactions))


Customers: 8000
Accounts: 9958
Transactions: 1000000


In [1]:
import pandas as pd
df_customers = pd.read_csv(r"E:\VS code stuff\Banks Data\DB_bank\customers.csv")
df_accounts = pd.read_csv(r"E:\VS code stuff\Banks Data\DB_bank\accounts.csv")
df_transactions = pd.read_csv(r"E:\VS code stuff\Banks Data\DB_bank\transactions.csv")
df_strs = pd.read_csv(r"E:\VS code stuff\Banks Data\DB_bank\str.csv")

In [5]:
import pandas as pd

df = pd.DataFrame({
    "transaction_id": ["T1", "T2", "T3", "T4"],
    "account_id": ["ACC_000572"] * 4,
    "transaction_datetime": [
        "2024-07-01 10:00",
        "2024-07-02 12:00",
        "2024-07-04 09:00",
        "2024-07-07 18:00",
    ],
    "transaction_category": ["CASH"] * 4,
    "transaction_amount": [10000, 15000, 20000, 5000],
})


In [6]:
df["transaction_datetime"] = pd.to_datetime(df["transaction_datetime"])
df = df.sort_values("transaction_datetime")


In [7]:
df_acc = df[df["account_id"] == "ACC_000572"]


In [8]:
df_acc["cash_1d_sum"] = (
    df_acc
        .set_index("transaction_datetime")
        .rolling("1D")["transaction_amount"]
        .sum()
        .values
)


In [11]:
df_acc

,transaction_id,account_id,transaction_datetime,transaction_category,transaction_amount,cash_1d_sum,cash_5d_sum
0,T1,ACC_000572,2024-07-01 10:00:00,CASH,10000,10000.0,10000.0
1,T2,ACC_000572,2024-07-02 12:00:00,CASH,15000,15000.0,25000.0
2,T3,ACC_000572,2024-07-04 09:00:00,CASH,20000,20000.0,45000.0
3,T4,ACC_000572,2024-07-07 18:00:00,CASH,5000,5000.0,25000.0


In [10]:
df_acc["cash_5d_sum"] = (
    df_acc
        .set_index("transaction_datetime")
        .rolling("5D")["transaction_amount"]
        .sum()
        .values
)


In [4]:
df_transactions.loc[0]

transaction_id          TXN_00000000
account_id                ACC_002137
customer_id               CUST_01705
transaction_datetime      2024-07-07
transaction_type               DEBIT
transaction_category            RTGS
transaction_amount         697775.69
Name: 0, dtype: object

In [3]:
df_accounts

,account_id,customer_id,account_type,account_open_date,account_status,dormancy_flag,last_dormant_date,account_close_date
0,ACC_000001,CUST_00001,SAVINGS,1979-10-02,ACTIVE,N,NaN,NaN
1,ACC_000002,CUST_00002,SAVINGS,1991-08-25,ACTIVE,N,NaN,NaN
2,ACC_000003,CUST_00002,SAVINGS,2007-11-06,ACTIVE,N,NaN,NaN
3,ACC_000004,CUST_00003,SAVINGS,2001-04-27,ACTIVE,N,NaN,NaN
4,ACC_000005,CUST_00003,SAVINGS,1971-05-25,ACTIVE,N,NaN,NaN
...,...,...,...,...,...,...,...,...
9974,ACC_009975,CUST_07997,SAVINGS,1952-04-19,ACTIVE,N,NaN,NaN
9975,ACC_009976,CUST_07998,SAVINGS,1964-06-11,ACTIVE,N,NaN,NaN
9976,ACC_009977,CUST_07998,SAVINGS,1989-09-30,ACTIVE,N,NaN,NaN
9977,ACC_009978,CUST_07999,CURRENT,1984-07-15,ACTIVE,N,NaN,NaN


In [4]:
df_customers

,customer_id,customer_type,gender,income_bracket,kyc_status,pep_flag,sanction_flag,internal_watchlist_flag,customer_risk_rating,dob_or_incorporation_date,customer_segment
0,CUST_00001,INDIVIDUAL,F,<5L,KYC_COMPLETE,N,N,N,LOW,1950-03-09,RETAIL
1,CUST_00002,NPO,F,<5L,KYC_COMPLETE,Y,N,N,HIGH,1938-06-09,RETAIL
2,CUST_00003,INDIVIDUAL,M,<5L,KYC_COMPLETE,N,N,N,MEDIUM,1972-06-18,RETAIL
3,CUST_00004,INDIVIDUAL,F,<5L,PENDING,N,N,N,MEDIUM,1968-05-12,RETAIL
4,CUST_00005,INDIVIDUAL,F,>25L,EXPIRED,N,N,N,LOW,1965-05-30,RETAIL
...,...,...,...,...,...,...,...,...,...,...,...
7995,CUST_07996,INDIVIDUAL,M,10-25L,KYC_COMPLETE,N,N,N,LOW,1964-11-28,RETAIL
7996,CUST_07997,INDIVIDUAL,F,<5L,KYC_COMPLETE,N,N,N,LOW,1962-08-30,RETAIL
7997,CUST_07998,INDIVIDUAL,F,<5L,KYC_COMPLETE,N,N,N,LOW,1996-07-24,RETAIL
7998,CUST_07999,CORPORATE,F,<5L,KYC_COMPLETE,N,N,N,LOW,1966-12-07,CORPORATE


In [61]:
tx_universe = df_transactions.copy()

tx_universe["transaction_datetime"] = pd.to_datetime(
    tx_universe["transaction_datetime"]
)

tx_universe = tx_universe[
    (tx_universe["transaction_category"] == "CASH") &
    (tx_universe["transaction_amount"] > 0)
]

print(f"Transactions in universe: {len(tx_universe):,}")
print(f"Unique accounts: {tx_universe['account_id'].nunique():,}")
print(f"Unique customers: {tx_universe['customer_id'].nunique():,}")


Transactions in universe: 12,425
Unique accounts: 6,567
Unique customers: 5,662


In [62]:
tx_universe

,transaction_id,account_id,customer_id,transaction_datetime,transaction_type,transaction_category,transaction_amount
56,TXN_00000056,ACC_008210,CUST_06566,2024-11-21,DEBIT,CASH,47307.50
57,TXN_00000057,ACC_002838,CUST_02273,2024-11-17,CREDIT,CASH,29542.97
61,TXN_00000061,ACC_003376,CUST_02713,2024-10-14,CREDIT,CASH,39336.94
92,TXN_00000092,ACC_008377,CUST_06695,2024-11-02,CREDIT,CASH,33774.81
112,TXN_00000112,ACC_007268,CUST_05811,2024-09-25,CREDIT,CASH,9910.15
...,...,...,...,...,...,...,...
249908,TXN_00249908,ACC_008214,CUST_06570,2024-07-17,DEBIT,CASH,26489.83
249911,TXN_00249911,ACC_004440,CUST_03550,2024-07-18,DEBIT,CASH,20694.72
249984,TXN_00249984,ACC_001022,CUST_00819,2024-09-15,DEBIT,CASH,17066.74
249986,TXN_00249986,ACC_008869,CUST_07104,2024-07-15,DEBIT,CASH,18864.59


In [67]:
BEHAVIOUR_CONFIG = {
    "entity_level": "account",      # "account" | "customer"
    "entity_id_col": "account_id",
    "time_col": "transaction_datetime",

    "metrics": [
        {
            "name": "cash_1d_sum",
            "type": "SUM",
            "column": "transaction_amount",
            "window": "1D",
            "frequency": "D"
        }
    ]
}


In [68]:
tx = tx_universe.copy()

tx["transaction_datetime"] = pd.to_datetime(tx["transaction_datetime"])
tx = tx.sort_values(
    [BEHAVIOUR_CONFIG["entity_id_col"],
     BEHAVIOUR_CONFIG["time_col"]]
)


In [73]:
tx['transaction_datetime'] = pd.to_datetime(tx['transaction_datetime'])

In [74]:
tx.dtypes

transaction_id                  object
account_id                      object
customer_id                     object
transaction_datetime    datetime64[ns]
transaction_type                object
transaction_category            object
transaction_amount             float64
dtype: object

In [77]:
def compute_behaviour(tx, config):
    entity = config["entity_id_col"]
    time_col = config["time_col"]

    behaviour_frames = []

    for metric in config["metrics"]:
        print(f"\n[STEP-2] Computing metric: {metric['name']}")

        if metric["type"] == "SUM":
            values = (
                tx
                .groupby(entity)
                .rolling(
                    metric["window"],
                    on=time_col
                )[metric["column"]]
                .sum()
                .reset_index(level=0, drop=True)
            )

        elif metric["type"] == "COUNT":
            values = (
                tx
                .groupby(entity)
                .rolling(
                    metric["window"],
                    on=time_col
                )[metric["column"]]
                .count()
                .reset_index(level=0, drop=True)
            )

        elif metric["type"] == "MAX":
            values = (
                tx
                .groupby(entity)
                .rolling(
                    metric["window"],
                    on=time_col
                )[metric["column"]]
                .max()
                .reset_index(level=0, drop=True)
            )

        else:
            raise ValueError(f"Unsupported metric type: {metric['type']}")

        behaviour_frames.append(
            pd.DataFrame({
                entity: tx[entity].values,
                "as_of_date": tx[time_col].values,
                "metric_name": metric["name"],
                "metric_value": values.values
            })
        )

    return pd.concat(behaviour_frames, ignore_index=True)


In [78]:
print(type(tx))
print(tx.columns.tolist())
print(tx["transaction_datetime"].dtype)


<class 'pandas.core.frame.DataFrame'>
['transaction_id', 'account_id', 'customer_id', 'transaction_datetime', 'transaction_type', 'transaction_category', 'transaction_amount']
datetime64[ns]


In [79]:
behaviour_table = compute_behaviour(tx, BEHAVIOUR_CONFIG)



[STEP-2] Computing metric: cash_1d_sum


In [80]:
behaviour_table

,account_id,as_of_date,metric_name,metric_value
0,ACC_000001,2024-07-01,cash_1d_sum,44281.22
1,ACC_000001,2024-09-21,cash_1d_sum,26207.81
2,ACC_000001,2024-11-28,cash_1d_sum,18367.67
3,ACC_000001,2024-12-01,cash_1d_sum,44565.63
4,ACC_000001,2024-12-07,cash_1d_sum,34297.13
...,...,...,...,...
12420,ACC_009977,2024-09-03,cash_1d_sum,10366.23
12421,ACC_009977,2024-11-14,cash_1d_sum,44808.51
12422,ACC_009977,2024-12-04,cash_1d_sum,32723.33
12423,ACC_009978,2024-10-22,cash_1d_sum,8325.14


In [93]:
print("\n[STEP-3.1] Entity-level worst behaviour")

entity_behaviour = (
    behaviour_table
    .groupby("account_id", as_index=False)
    .agg(
        worst_value=("metric_value", "max"),
        last_seen_date=("as_of_date", "max")
    )
)

print("Entities:", entity_behaviour["account_id"].nunique())



[STEP-3.1] Entity-level worst behaviour
Entities: 6567


In [94]:
print("\n[STEP-3.2] Behaviour distribution")

print(
    entity_behaviour["worst_value"]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.97, 0.99])
)



[STEP-3.2] Behaviour distribution
count     6567.000000
mean     31431.695747
std      13635.483725
min       1012.990000
50%      33695.840000
90%      47285.382000
95%      48710.522000
97%      49349.890000
99%      49841.059200
max      96197.770000
Name: worst_value, dtype: float64


In [95]:
print("\n[STEP-3.3] Threshold simulation")

percentiles = [0.90, 0.95, 0.97, 0.99]
total_entities = len(entity_behaviour)

threshold_rows = []

for p in percentiles:
    thr = entity_behaviour["worst_value"].quantile(p)
    cnt = (entity_behaviour["worst_value"] > thr).sum()

    threshold_rows.append({
        "percentile": int(p * 100),
        "threshold": round(thr, 2),
        "alerts": cnt,
        "alert_pct": round(cnt / total_entities * 100, 2)
    })

threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df)



[STEP-3.3] Threshold simulation
   percentile  threshold  alerts  alert_pct
0          90   47285.38     657      10.00
1          95   48710.52     329       5.01
2          97   49349.89     197       3.00
3          99   49841.06      66       1.01


In [96]:
print("\n[STEP-3.4] Selecting threshold")

CHOSEN_PCT = 0.95
THRESHOLD = entity_behaviour["worst_value"].quantile(CHOSEN_PCT)

print(f"Chosen threshold ({int(CHOSEN_PCT*100)}th): {THRESHOLD:,.2f}")



[STEP-3.4] Selecting threshold
Chosen threshold (95th): 48,710.52


In [98]:
print("\n[STEP-3.5] Labeling ATL / BTL at entity level")

entity_behaviour["ATL_FLAG"] = (
    entity_behaviour["worst_value"] > THRESHOLD
)

print(entity_behaviour["ATL_FLAG"].value_counts())



[STEP-3.5] Labeling ATL / BTL at entity level
ATL_FLAG
False    6238
True      329
Name: count, dtype: int64


In [99]:
print("\n[STEP-3.6] Mapping entity labels back to behaviour rows")

behaviour_labeled = behaviour_table.merge(
    entity_behaviour[["account_id", "ATL_FLAG"]],
    on="account_id",
    how="left"
)



[STEP-3.6] Mapping entity labels back to behaviour rows


In [100]:
from scipy.stats import ks_2samp

print("\n[STEP-3.7] KS on behaviour rows")

atl_vals = behaviour_labeled.loc[
    behaviour_labeled["ATL_FLAG"], "metric_value"
]

btl_vals = behaviour_labeled.loc[
    ~behaviour_labeled["ATL_FLAG"], "metric_value"
]

ks_stat, p_val = ks_2samp(atl_vals, btl_vals)

print(f"KS statistic : {ks_stat:.4f}")
print(f"P-value      : {p_val:.4e}")



[STEP-3.7] KS on behaviour rows
KS statistic : 0.4009
P-value      : 8.5700e-114


In [101]:
print("\n[STEP-3.8] Overlap check")

print("ATL min:", atl_vals.min())
print("BTL max:", btl_vals.max())



[STEP-3.8] Overlap check
ATL min: 1135.39
BTL max: 48708.38


STEP-4 IMPLEMENTATION 

In [102]:
print("\n[STEP-4.1] Selecting ATL entities only")

atl_entities = entity_behaviour[
    entity_behaviour["ATL_FLAG"]
].copy()

print("ATL entities:", len(atl_entities))



[STEP-4.1] Selecting ATL entities only
ATL entities: 329


In [103]:
print("\n[STEP-4.2] Joining account and customer master data")

alerts_base = (
    atl_entities
    .merge(df_accounts, on="account_id", how="left")
    .merge(df_customers, on="customer_id", how="left")
)

print("Rows after joins:", len(alerts_base))



[STEP-4.2] Joining account and customer master data
Rows after joins: 329


In [104]:
print("\n[STEP-4.3] Applying eligibility rules")

eligible_alerts = alerts_base[
    (alerts_base["account_status"] == "ACTIVE") &
    (alerts_base["pep_flag"] == "N") &
    (~alerts_base["sanction_flag"].eq("Y"))
]

print("Eligible alerts:", len(eligible_alerts))



[STEP-4.3] Applying eligibility rules
Eligible alerts: 326


In [105]:
print("\n[STEP-4.4] Creating alert records")

eligible_alerts["alert_id"] = (
    "ALERT_" + eligible_alerts["account_id"]
)

eligible_alerts["alert_date"] = (
    eligible_alerts["last_seen_date"]
)

eligible_alerts["scenario_name"] = "CASH_30D_SUM"
eligible_alerts["scenario_threshold"] = THRESHOLD
eligible_alerts["alert_status"] = "OPEN"



[STEP-4.4] Creating alert records


C:\Users\kriss\AppData\Local\Temp\ipykernel_7048\1769812381.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eligible_alerts["alert_id"] = (
C:\Users\kriss\AppData\Local\Temp\ipykernel_7048\1769812381.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eligible_alerts["alert_date"] = (
C:\Users\kriss\AppData\Local\Temp\ipykernel_7048\1769812381.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

In [106]:
alert_table = eligible_alerts[
    [
        "alert_id",
        "account_id",
        "customer_id",
        "alert_date",
        "scenario_name",
        "scenario_threshold",
        "worst_value",
        "alert_status"
    ]
]

print("\n[STEP-4.5] Final alerts generated")
print(alert_table.head())
print("Total alerts:", len(alert_table))



[STEP-4.5] Final alerts generated
           alert_id  account_id customer_id alert_date scenario_name  \
0  ALERT_ACC_000078  ACC_000078  CUST_00064 2024-11-16  CASH_30D_SUM   
1  ALERT_ACC_000108  ACC_000108  CUST_00088 2024-12-10  CASH_30D_SUM   
2  ALERT_ACC_000117  ACC_000117  CUST_00094 2024-12-02  CASH_30D_SUM   
3  ALERT_ACC_000143  ACC_000143  CUST_00115 2024-09-15  CASH_30D_SUM   
4  ALERT_ACC_000156  ACC_000156  CUST_00127 2024-11-20  CASH_30D_SUM   

   scenario_threshold  worst_value alert_status  
0           48710.522     48958.44         OPEN  
1           48710.522     49884.77         OPEN  
2           48710.522     49025.20         OPEN  
3           48710.522     49659.80         OPEN  
4           48710.522     49916.33         OPEN  
Total alerts: 326


STEP-5

In [110]:
df_str = pd.read_csv(r"E:\VS code stuff\Banks Data\DB_bank\str.csv")

In [111]:
df_str["str_filed_date"] = pd.to_datetime(df_str["str_filed_date"])


In [113]:
print("\n[STEP-5.2] Mapping STRs to alerts")

alerts_with_str = alert_table.merge(
    df_str,
    on="account_id",
    how="left"
)



[STEP-5.2] Mapping STRs to alerts


In [114]:
print("\n[STEP-5.3] Applying temporal alignment")

alerts_with_str["is_str"] = (
    (alerts_with_str["str_filed_date"].notna()) &
    (alerts_with_str["alert_date"] <= alerts_with_str["str_filed_date"])
).astype(int)



[STEP-5.3] Applying temporal alignment


In [115]:
print("\n[STEP-5.4] STR sanity checks")

print("Total alerts:", len(alerts_with_str))
print("STR alerts:", alerts_with_str["is_str"].sum())
print("STR rate (%):", alerts_with_str["is_str"].mean() * 100)



[STEP-5.4] STR sanity checks
Total alerts: 330
STR alerts: 15
STR rate (%): 4.545454545454546


In [116]:
print("\n[STEP-5.5] STR accounts never alerted")

str_accounts = set(df_str["account_id"])
alerted_accounts = set(alerts_with_str["account_id"])

missed_accounts = str_accounts - alerted_accounts

print("Missed STR accounts:", len(missed_accounts))



[STEP-5.5] STR accounts never alerted
Missed STR accounts: 248


In [117]:
print("\n[STEP-5.6] Threshold vs STR capture")

sensitivity = []

for _, row in threshold_df.iterrows():
    thr = row["threshold"]

    subset = alerts_with_str[
        alerts_with_str["worst_value"] > thr
    ]

    sensitivity.append({
        "threshold": thr,
        "alerts": len(subset),
        "str_captured": subset["is_str"].sum(),
        "str_missed": alerts_with_str["is_str"].sum() - subset["is_str"].sum()
    })

pd.DataFrame(sensitivity)



[STEP-5.6] Threshold vs STR capture


,threshold,alerts,str_captured,str_missed
0,47285.38,330,15,0
1,48710.52,330,15,0
2,49349.89,199,12,3
3,49841.06,66,4,11
